# 05 — Random Forest Training

This notebook trains a Random Forest classifier for multi-class DoS attack classification:
- Load preprocessed labeled data
- Split into train / test sets
- Train the Random Forest model
- Evaluate with classification report, confusion matrix, and feature importance
- Persist the trained model to disk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)

sys.path.insert(0, os.path.join(os.path.dirname("__file__" if "__file__" in dir() else ""), ".."))
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Load Labeled Data

In [ ]:
FEATURES_PATH = os.path.join("..", "data", "processed", "features_scaled.npy")
LABELS_PATH = os.path.join("..", "data", "processed", "labels.npy")
META_PATH = os.path.join("..", "data", "processed", "feature_columns.npy")

if os.path.exists(FEATURES_PATH):
    X = np.load(FEATURES_PATH)
    y = np.load(LABELS_PATH)
    feature_names = np.load(META_PATH, allow_pickle=True).tolist()
else:
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    RAW_PATH = os.path.join("..", "data", "raw", "CICIDS2017", "Wednesday-workingHours.pcap_ISCX.csv")
    df = pd.read_csv(RAW_PATH).drop_duplicates().reset_index(drop=True)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["Label"]).reset_index(drop=True)
    META_COLS = ["Flow ID", "Source IP", "Source Port", "Destination IP",
                 "Destination Port", "Protocol", "Timestamp", "Label"]
    feature_names = [c for c in df.columns if c not in META_COLS and df[c].dtype in ["float64", "int64"]]
    X_raw = df[feature_names].replace([np.inf, -np.inf], np.nan).fillna(0).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    le = LabelEncoder()
    y = le.fit_transform(df["Label"])

print(f"Features: {X.shape}")
print(f"Labels:   {y.shape}")
print(f"Classes:  {np.unique(y)}")

## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train:  {X_train.shape[0]:,} samples")
print(f"Test:   {X_test.shape[0]:,} samples")
print(f"\nTrain class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt:,} ({cnt / len(y_train) * 100:.1f}%)")

## 3. Train Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print(f"Model trained: {rf_model.n_estimators} trees")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

## 4. Classification Report

In [ ]:
# Build label names from encoder if available
try:
    label_names = le.classes_
except NameError:
    label_names = [str(i) for i in sorted(np.unique(y))]

print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

## 5. Feature Importance

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 8))
top_names = [feature_names[i] if i < len(feature_names) else f"feat_{i}" for i in indices]
top_vals = importances[indices]

ax.barh(range(20), top_vals[::-1], color="steelblue")
ax.set_yticks(range(20))
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel("Importance")
ax.set_title("Top 20 Feature Importances — Random Forest")
plt.tight_layout()
plt.show()

## 6. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=label_names,
    cmap="Blues", ax=ax, xticks_rotation=45,
)
ax.set_title("Confusion Matrix — Random Forest")
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
MODEL_DIR = os.path.join("..", "models", "trained")
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "random_forest.pkl")
joblib.dump(rf_model, model_path)
print(f"Model saved to: {os.path.abspath(model_path)}")
print(f"File size: {os.path.getsize(model_path) / 1024:.1f} KB")